# Task 13: API & Error Handling
Call a real API (Open-Meteo weather, no key needed) and handle 404, 429 and 500 errors with try-except.

## 1. Call a weather API

In [1]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {"latitude": 26.91, "longitude": 75.79, "current_weather": True}  # Jaipur

response = requests.get(url, params=params, timeout=10)
data = response.json()
print("Status:", response.status_code)
print("Temperature:", data["current_weather"]["temperature"], "°C")
print("Wind speed:", data["current_weather"]["windspeed"], "km/h")

Status: 200
Temperature: 22.5 °C
Wind speed: 6.8 km/h


## 2. Reusable function with error handling

In [2]:
def call_api(url, params=None):
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()          # raises error for 4xx / 5xx
        return response.json()

    except requests.exceptions.HTTPError as e:
        code = e.response.status_code
        if code == 404:
            print("Error 404: Not Found. Check the URL or endpoint.")
        elif code == 429:
            print("Error 429: Rate limit hit. Wait a bit and try again.")
        elif code >= 500:
            print(f"Error {code}: Server problem. Try again later.")
        else:
            print(f"HTTP error {code}: {e}")
    except requests.exceptions.Timeout:
        print("Error: Request timed out.")
    except requests.exceptions.ConnectionError:
        print("Error: No internet connection.")
    except Exception as e:
        print(f"Unexpected error: {e}")
    return None

## 3. Test each error (httpbin returns any status code we ask for)

In [14]:
import requests

def call_api(url, params=None):
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        try:
            return response.json()
        except ValueError:
            print(f"Success {response.status_code}: request worked (no JSON in response).")
            return response.text
    except requests.exceptions.HTTPError as e:
        code = e.response.status_code
        if code == 404:
            print("Error 404: Not Found. Check the URL or endpoint.")
        elif code == 429:
            print("Error 429: Rate limit hit. Wait a bit and try again.")
        elif code >= 500:
            print(f"Error {code}: Server problem. Try again later.")
        else:
            print(f"HTTP error {code}: {e}")
    except requests.exceptions.Timeout:
        print("Error: Request timed out.")
    except requests.exceptions.ConnectionError:
        print("Error: No internet connection.")
    except Exception as e:
        print(f"Unexpected error: {e}")
    return None

for status in [200, 404, 429, 500]:
    print(f"--- Testing {status} ---")
    call_api(f"https://httpbin.org/status/{status}")

print("Done")

--- Testing 200 ---
Success 200: request worked (no JSON in response).
--- Testing 404 ---
Error 404: Not Found. Check the URL or endpoint.
--- Testing 429 ---
Error 429: Rate limit hit. Wait a bit and try again.
--- Testing 500 ---
Error 500: Server problem. Try again later.
Done


## 4. Bonus: retry on 429 / 500

In [3]:
import time

def call_with_retry(url, retries=3):
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code in (429, 500):
                print(f"Attempt {attempt}: got {r.status_code}, retrying...")
                time.sleep(2 ** attempt)     # wait 2s, 4s, 8s
                continue
            r.raise_for_status()
            return r.json()
        except requests.exceptions.RequestException as e:
            print("Error:", e)
            return None
    print("Failed after all retries.")

call_with_retry("https://httpbin.org/status/429")

Attempt 1: got 429, retrying...
Attempt 2: got 429, retrying...
Attempt 3: got 429, retrying...
Failed after all retries.
